In [1]:
import black
import jupyter_black

jupyter_black.load(
    lab=False,
    line_length=79,
    target_version=black.TargetVersion.PY39,
)

<IPython.core.display.Javascript object>

In [2]:
import numpy as np
from boolean_analysis import calculate_fourier_transform_matrix, parity_of_1s
import pandas as pd
from tqdm import tqdm
import pickle
from boolean_fourier_learner import BooleanFourierLearner
from pathlib import Path
import lzma
import parse
from spin_lattices import KagomeLattice
from misc_utils import make_unpacked_configurations
import matplotlib.pyplot as plt
from heisenberg_hamiltonians import batched_state_info_df, HeisenbergJ1J2
import lattice_symmetries as ls
import seaborn as sns

%matplotlib inline

In [4]:
J2 = 0.53


lat = KagomeLattice(width=2, height=4)
system = HeisenbergJ1J2(lat, J1=1, J2=J2, use_symmetries=True)
system.get_eigenstates(0)

fourier_basis = ls.SpinBasis(
    system.symmetry_group,
    number_spins=system.number_spins,
    hamming_weight=None,
    spin_inversion=None,
)
fourier_basis.build()
state_info_df = batched_state_info_df(
    fourier_basis, np.arange(2**system.number_spins, dtype="uint64")
)
state_info_df_system = (
    batched_state_info_df(system.basis, system.canonical_basis.states)
    .reset_index()
    .rename(columns={"index": "state"})
)

self.number_spins=24
Symmetry group contains 16 elements
Hilbert space dimension is 85662
Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-0.53-True-1-1.pickle
Ground state energy is -39.2199137373


In [30]:
learners = {}
for file in experiment_dir.glob("fourier-learner-*.pickle.lz"):
    J, i = parse.search("fourier-learner-{}-{:d}.pickle.lz", str(file)).fixed
    J = float(J)
    with lzma.open(file) as f:
        learners[J, i] = pickle.load(f)

learners_ser = (
    pd.DataFrame(dict(learner=learners))
    .reset_index()
    .rename(columns={"level_0": "J", "level_1": "iterations"})
).set_index(["J", "iterations"])["learner"]

for learner in learners.values():
    learner.coeffs_df_ = None

In [5]:
train_sets = {}
for file in experiment_dir.glob("train-*.feather"):
    (J,) = parse.search("train-{}.feather", str(file)).fixed
    J = float(J)
    train_sets[J] = pd.read_feather(file).set_index("index")

In [7]:
train = train_sets[J2]

In [8]:
train

,eigenstate_coeff,amplitude,prob
index,,,
9260721,0.006846,0.006846,4.686099e-05
14968205,-0.000833,0.000833,6.944457e-07
708307,0.004842,0.004842,2.344718e-05
1162397,0.000040,0.000040,1.627268e-09
6933045,-0.018528,0.018528,3.432886e-04
...,...,...,...
1759001,-0.003872,0.003872,1.499220e-05
8040564,0.000866,0.000866,7.503435e-07
7521843,-0.005525,0.005525,3.052909e-05


In [17]:
train_sign = np.sign(train["eigenstate_coeff"])

In [49]:
random_steps = 10000
print_each = 10
set_ = np.random.choice(np.arange(0, 2**system.number_spins, dtype="uint64"))
reset_mutations_each = 100


def get_score(set_):
    return (
        calculate_fourier_transform_matrix(
            np.array(train.index), np.array([set_]), system.number_spins
        )[:, 0]
        * train_sign
    ).mean()


mutations = 1

score = get_score(set_)
for i in range(random_steps):
    if i % reset_mutations_each == 0:
        mutations = np.random.randint(1, system.number_spins)
    new_set = set_
    for j in range(mutations):
        change_bit = np.random.randint(0, system.number_spins, dtype="uint64")
        new_set = new_set ^ (np.array(1, dtype="uint64") << change_bit)
    new_score = get_score(new_set)
    if new_score > score:
        set_ = new_set
        score = new_score
    if i % print_each == 0:
        print(score)

0.0115
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0178
0.0292
0.0292
0.0292
0.0292
0.0292
0.0292
0.0292
0.0292
0.0292
0.0292
0.0292
0.0292
0.0292
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0377
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0456
0.0671
0.0671
0.0671
0.0671
0.0671
0.0671
0.0671
0.0671
0.0671
0.0671
0.0671
0.0671
0.0671
0.0671
0.0671
0.0671
0.0671
0.0671

In [24]:
1 << 4

16

In [40]:
def expand_fourier_coeffs(coeffs, fourier_basis, number_spins):
    state_info_df = batched_state_info_df(
        fourier_basis, np.arange(2**number_spins, dtype="uint64")
    )
    return state_info_df.join(coeffs, on="representative")["coeff"].dropna()

In [41]:
coeffs = expand_fourier_coeffs(
    learners[J2, 200].get_coeffs_df(), fourier_basis, system.number_spins
)

In [50]:
coeffs.to_frame().sort_values("coeff", ascending=False).reset_index()[
    lambda x: x["index"] == set_
]

,index,coeff
277,10827342,0.1411
